# 🌫️ Delhi Hourly PM2.5 & PM10 — OpenAQ API v3

Fetches **station-wise hourly PM2.5 and PM10** data for Delhi from the [OpenAQ API v3](https://api.openaq.org/v3).

✅ Works on **Google Colab** and **local JupyterLab** — paths are auto-detected.  
✅ Sensors fetched via `/v3/locations/{id}/sensors` (robust to empty `sensors[]` in location list).  
✅ Rate-limit and server-error handling built in.

---
### API v3 Workflow
```
GET /v3/locations?parameters_id=2&bbox=...   ← PM2.5 locations in Delhi
GET /v3/locations?parameters_id=1&bbox=...   ← PM10  locations in Delhi
GET /v3/locations/{id}/sensors               ← sensors for each location
GET /v3/sensors/{sensor_id}/hours            ← hourly averages
```
**Parameter IDs:** PM2.5 = `2`, PM10 = `1`

## Cell 1 — Install dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg}…")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else:
        print(f"✅ {pkg} already installed")

ensure('requests')
ensure('pandas')
ensure('tqdm')

## Cell 2 — Configuration ⚙️

> ⚠️ **API_KEY must be a string — wrap it in quotes:** `API_KEY = "abc123..."`

In [ ]:
import requests
import pandas as pd
import time
import os
import platform
from datetime import datetime, timezone
from tqdm.notebook import tqdm

# ──────────────────────────────────────────────────────────────────
#  ✏️  EDIT HERE
# ──────────────────────────────────────────────────────────────────
API_KEY   = "974d17904d669e71bbf5f90d8d2c81bd638b3f619c581f47c2c8be270ea411dd"   # ← paste between the quotes

DATE_FROM = "2026-01-01T00:00:00Z"
DATE_TO   = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")  # today

# ──────────────────────────────────────────────────────────────────
#  OUTPUT DIRECTORY — auto-detected
# ──────────────────────────────────────────────────────────────────
def detect_output_dir():
    if os.path.isdir("/content/drive/MyDrive"):
        out = "/content/drive/MyDrive/Delhi_OpenAQ"
    elif os.path.isdir("/content"):
        out = "/content/Delhi_OpenAQ"
    else:
        out = os.path.join(os.getcwd(), "Delhi_OpenAQ_Output")
    os.makedirs(out, exist_ok=True)
    return out

OUTPUT_DIR = detect_output_dir()
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "delhi_pm25_pm10_hourly_openaq.csv")

# ──────────────────────────────────────────────────────────────────
#  API SETTINGS
# ──────────────────────────────────────────────────────────────────
BASE_URL      = "https://api.openaq.org/v3"
HEADERS       = {"X-API-Key": API_KEY}
PARAMS_BATCH  = 1000
RETRY_MAX     = 3
SLEEP_SEC     = 1.2    # ~50 req/min — safely under free-tier 60/min limit
SLEEP_429     = 30     # seconds to back off on rate-limit

# OpenAQ parameter IDs  (confirmed: PM2.5=2, PM10=1)
PARAM_IDS = {"pm25": 2, "pm10": 1}

# Delhi bounding box: minLon, minLat, maxLon, maxLat
DELHI_BBOX = "76.84,28.40,77.35,28.89"

print(f"Platform   : {platform.system()}")
print(f"Date range : {DATE_FROM}  →  {DATE_TO}")
print(f"Output dir : {OUTPUT_DIR}")

## Cell 3 — Helper functions

In [ ]:
def safe_get(url, params=None, attempt=1):
    """
    GET with smart retry:
      429 → wait SLEEP_429 s, retry up to RETRY_MAX
      500/502/503 → skip gracefully after RETRY_MAX
    Returns parsed JSON dict or {} on failure.
    """
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=45)

        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", SLEEP_429))
            wait = max(wait, SLEEP_429)
            print(f"  ⏳ 429 Rate-limit — waiting {wait}s (attempt {attempt}/{RETRY_MAX})")
            time.sleep(wait)
            if attempt <= RETRY_MAX:
                return safe_get(url, params, attempt + 1)
            return {}

        if r.status_code in (500, 502, 503):
            if attempt <= RETRY_MAX:
                time.sleep(5 * attempt)
                return safe_get(url, params, attempt + 1)
            # Gracefully skip — no data for this sensor/period on OpenAQ server
            return {}

        r.raise_for_status()
        return r.json()

    except requests.exceptions.Timeout:
        if attempt <= RETRY_MAX:
            time.sleep(5 * attempt)
            return safe_get(url, params, attempt + 1)
        return {}
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {e}")
        return {}


def paginate(url, params):
    """Fetch all pages of a paginated endpoint."""
    results, page = [], 1
    while True:
        data  = safe_get(url, {**params, "page": page})
        batch = data.get("results", [])
        results.extend(batch)
        if len(batch) < params.get("limit", 100):
            break
        page += 1
        time.sleep(SLEEP_SEC)
    return results


def month_chunks(date_from_str, date_to_str):
    """Split date range into monthly windows to avoid API timeouts."""
    fmt   = "%Y-%m-%dT%H:%M:%SZ"
    start = datetime.strptime(date_from_str, fmt).replace(tzinfo=timezone.utc)
    end   = datetime.strptime(date_to_str,   fmt).replace(tzinfo=timezone.utc)
    chunks, cur = [], start
    while cur < end:
        nxt = cur.replace(month=cur.month % 12 + 1, day=1)
        if cur.month == 12:
            nxt = nxt.replace(year=cur.year + 1)
        nxt = min(nxt, end)
        chunks.append((cur.strftime(fmt), nxt.strftime(fmt)))
        cur = nxt
    return chunks


print("✅ Helper functions defined")

## Cell 4 — 🔬 Diagnostic: Inspect raw API response

This cell shows exactly what the OpenAQ API returns for Delhi — run it first to confirm connectivity and structure.

In [ ]:
import json

# ── Test 1: Fetch a single PM2.5 location in Delhi ──────────────
test_url    = f"{BASE_URL}/locations"
test_params = {
    "bbox"          : DELHI_BBOX,
    "parameters_id" : PARAM_IDS["pm25"],   # 2
    "limit"         : 1,
    "page"          : 1,
}
resp = safe_get(test_url, test_params)

meta = resp.get("meta", {})
print("── API /v3/locations (PM2.5, limit=1) ──")
print(f"  found   : {meta.get('found', 'N/A')}")
print(f"  page    : {meta.get('page', 'N/A')}")
print(f"  results : {len(resp.get('results', []))} record(s) returned")
print()

results = resp.get("results", [])
if results:
    loc = results[0]
    print("── First location keys:", list(loc.keys()))
    print(f"  id      : {loc.get('id')}")
    print(f"  name    : {loc.get('name')}")
    print(f"  sensors : {loc.get('sensors', '(not in response)')}")
    print()

    # ── Test 2: Fetch sensors for this location ─────────────────
    loc_id      = loc.get("id")
    sensors_url = f"{BASE_URL}/locations/{loc_id}/sensors"
    sresp       = safe_get(sensors_url)
    sensors     = sresp.get("results", [])
    print(f"── /v3/locations/{loc_id}/sensors → {len(sensors)} sensor(s)")
    for s in sensors:
        param = s.get("parameter", {})
        print(f"   sensor_id={s.get('id')}  param_name={param.get('name')}  "
              f"param_id={param.get('id')}  units={param.get('units')}")
else:
    print("⚠️  No locations returned — check your API key and bbox.")
    print("   Raw response snippet:")
    print(json.dumps(resp, indent=2)[:800])

## Cell 5 — Discover All Delhi PM2.5 & PM10 Stations

Queries locations separately for PM2.5 (`parameters_id=2`) and PM10 (`parameters_id=1`), then fetches sensors for each location via `/v3/locations/{id}/sensors`.

In [ ]:
sensor_rows = []

for param_name, param_id in PARAM_IDS.items():
    print(f"\n🔍 Fetching {param_name.upper()} locations (parameters_id={param_id}) …")

    raw_locs = paginate(
        f"{BASE_URL}/locations",
        {"bbox": DELHI_BBOX, "parameters_id": param_id, "limit": 100, "iso": "IN"}
    )
    print(f"   → {len(raw_locs)} location(s) found")

    for loc in raw_locs:
        loc_id   = loc.get("id")
        loc_name = loc.get("name", "Unknown")
        coords   = loc.get("coordinates") or {}
        lat      = coords.get("latitude")
        lon      = coords.get("longitude")
        provider = (loc.get("provider") or {}).get("name", "")
        country  = (loc.get("country")  or {}).get("code", "")

        # ── Fetch sensors for this location ─────────────────────
        sensors_url  = f"{BASE_URL}/locations/{loc_id}/sensors"
        sensors_resp = safe_get(sensors_url)
        sensors      = sensors_resp.get("results", [])
        time.sleep(SLEEP_SEC)

        for sensor in sensors:
            p      = sensor.get("parameter") or {}
            p_name = (p.get("name") or "").lower().replace(".", "").replace(" ", "")  # normalise
            p_id   = p.get("id")

            # Match by parameter_id (most reliable) OR normalised name
            if p_id not in (1, 2) and p_name not in ("pm25", "pm10"):
                continue

            # Resolve canonical name from id
            canonical = "pm25" if p_id == 2 else ("pm10" if p_id == 1 else p_name)

            # Skip if this sensor doesn't match the parameter we're looking for
            if canonical != param_name:
                continue

            sensor_rows.append({
                "location_id"  : loc_id,
                "station_name" : loc_name,
                "provider"     : provider,
                "country"      : country,
                "latitude"     : lat,
                "longitude"    : lon,
                "sensor_id"    : sensor.get("id"),
                "parameter"    : canonical,
                "unit"         : p.get("units", "µg/m³"),
            })

# ── Build sensors DataFrame ───────────────────────────────────────
if sensor_rows:
    sensors_df = pd.DataFrame(sensor_rows).drop_duplicates(subset="sensor_id").reset_index(drop=True)
    print(f"\n✅ Total sensors found (PM2.5 + PM10): {len(sensors_df)}")
    print(f"   Unique stations : {sensors_df['station_name'].nunique()}")
    print(f"   PM2.5 sensors   : {(sensors_df.parameter == 'pm25').sum()}")
    print(f"   PM10  sensors   : {(sensors_df.parameter == 'pm10').sum()}")
    display(sensors_df[['station_name','provider','parameter','sensor_id','latitude','longitude']])
else:
    sensors_df = pd.DataFrame()
    print("\n⚠️  No PM2.5 / PM10 sensors found.")
    print("   Possible reasons:")
    print("     1. API key invalid or expired")
    print("     2. No stations active in the Delhi bounding box")
    print("     3. Run Cell 4 (Diagnostic) first to inspect the raw response")

## Cell 6 — Fetch Hourly Data for Every Sensor

> `SLEEP_SEC=1.2` keeps requests at ~50/min (free-tier limit: 60/min).  
> HTTP 500 errors are **skipped gracefully** — OpenAQ has no data for that sensor/period.

In [ ]:
if sensors_df.empty:
    print("⚠️  sensors_df is empty — run Cells 4 and 5 first.")
else:
    chunks = month_chunks(DATE_FROM, DATE_TO)
    print(f"Date range : {len(chunks)} monthly chunk(s)")
    print(f"Sensors    : {len(sensors_df)}")
    print(f"Total API calls (est.) : {len(sensors_df) * len(chunks)}\n")

    all_records   = []
    skipped_count = 0

    for _, row in tqdm(sensors_df.iterrows(), total=len(sensors_df), desc="Sensors"):
        sensor_url = f"{BASE_URL}/sensors/{row.sensor_id}/hours"

        for (dfrom, dto) in chunks:
            data    = safe_get(sensor_url, {"date_from": dfrom, "date_to": dto, "limit": PARAMS_BATCH})
            records = data.get("results", [])

            if not records:
                skipped_count += 1

            for rec in records:
                period       = rec.get("period") or {}
                dt_from_info = period.get("datetimeFrom") or {}
                dt_to_info   = period.get("datetimeTo")   or {}
                summary      = rec.get("summary") or {}
                all_records.append({
                    "location_id"     : row.location_id,
                    "station_name"    : row.station_name,
                    "provider"        : row.provider,
                    "latitude"        : row.latitude,
                    "longitude"       : row.longitude,
                    "sensor_id"       : row.sensor_id,
                    "parameter"       : row.parameter,
                    "unit"            : row.unit,
                    "datetime_utc"    : dt_from_info.get("utc"),
                    "datetime_local"  : dt_from_info.get("local"),
                    "datetime_to_utc" : dt_to_info.get("utc"),
                    "value"           : rec.get("value"),
                    "min"             : summary.get("min"),
                    "max"             : summary.get("max"),
                    "sd"              : summary.get("sd"),
                    "count"           : summary.get("count"),
                })

            time.sleep(SLEEP_SEC)

    print(f"\n✅ Total hourly records collected : {len(all_records):,}")
    print(f"   Empty sensor-months skipped   : {skipped_count}")

## Cell 7 — Build DataFrame & Preview

In [ ]:
df = pd.DataFrame(all_records)

if df.empty:
    print("⚠️  No data returned. Check your API key and date range.")
else:
    df["datetime_utc"]    = pd.to_datetime(df["datetime_utc"],    utc=True,  errors="coerce")
    df["datetime_local"]  = pd.to_datetime(df["datetime_local"],  utc=False, errors="coerce")
    df["datetime_to_utc"] = pd.to_datetime(df["datetime_to_utc"], utc=True,  errors="coerce")
    df["datetime_ist"]    = df["datetime_utc"].dt.tz_convert("Asia/Kolkata")

    df = df.sort_values(["station_name", "parameter", "datetime_utc"]).reset_index(drop=True)

    print(f"Shape      : {df.shape}")
    print(f"Date range : {df.datetime_utc.min()}  →  {df.datetime_utc.max()}")
    print(f"Stations   : {df.station_name.nunique()}")
    print()
    display(df[["station_name","provider","parameter","datetime_ist","value","unit","min","max","count"]].head(20))

## Cell 8 — Summary Statistics by Station & Parameter

In [ ]:
if not df.empty:
    summary = (
        df.groupby(["station_name", "parameter"])["value"]
          .agg(records="count", mean="mean", median="median", min="min", max="max", std="std")
          .round(2)
          .sort_values(["parameter", "station_name"])
          .reset_index()
    )
    print("Station-wise summary (µg/m³):")
    display(summary)

## Cell 9 — Pivot to Wide Format (one column per station)

In [ ]:
if not df.empty:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for param in ["pm25", "pm10"]:
        sub = df[df.parameter == param].copy()
        if sub.empty:
            print(f"No {param} data.")
            continue

        pivot = sub.pivot_table(
            index="datetime_ist", columns="station_name",
            values="value", aggfunc="mean"
        ).sort_index()

        print(f"\n{'='*60}")
        print(f"  {param.upper()} Pivot — rows=hours (IST), cols=stations")
        print(f"  Shape: {pivot.shape}")
        display(pivot.head(10))

        pivot_path = os.path.join(OUTPUT_DIR, f"delhi_{param}_hourly_pivot.csv")
        pivot.to_csv(pivot_path)
        print(f"  ✅ Saved → {pivot_path}")

        try:
            from google.colab import files
            files.download(pivot_path)
            print(f"  ⬇️  Colab download triggered")
        except ImportError:
            pass

## Cell 10 — Save Main Long-Format CSV

In [ ]:
if not df.empty:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cols = [
        "station_name", "provider", "location_id", "sensor_id",
        "latitude", "longitude", "parameter", "unit",
        "datetime_ist", "datetime_utc", "datetime_to_utc",
        "value", "min", "max", "sd", "count"
    ]
    cols = [c for c in cols if c in df.columns]
    df[cols].to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Saved {len(df):,} records → {OUTPUT_CSV}")

    try:
        from google.colab import files
        files.download(OUTPUT_CSV)
        print("⬇️  Colab download triggered")
    except ImportError:
        pass
else:
    print("⚠️  DataFrame is empty — nothing saved.")

## Cell 11 — Quick Visual: Daily Mean PM2.5 by Station

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

matplotlib.rcParams['figure.dpi'] = 120

if not df.empty:
    pm25 = df[df.parameter == "pm25"].copy()
    if not pm25.empty:
        pm25["date"] = pm25["datetime_ist"].dt.date
        daily = pm25.groupby(["date", "station_name"])["value"].mean().reset_index()
        daily["date"] = pd.to_datetime(daily["date"])

        top_stations = (
            daily.groupby("station_name")["value"].mean()
                 .nlargest(10).index.tolist()
        )
        fig, ax = plt.subplots(figsize=(14, 6))
        for station in top_stations:
            sub = daily[daily.station_name == station]
            ax.plot(sub["date"], sub["value"], lw=0.8, label=station)

        ax.axhline(15, color='green',  ls='--', lw=0.8, label="WHO 24h guideline (15 µg/m³)")
        ax.axhline(60, color='orange', ls='--', lw=0.8, label="NAAQS 24h (60 µg/m³)")
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.xticks(rotation=45, ha='right')
        ax.set_title("Delhi — Daily Mean PM2.5 (µg/m³) by Station", fontsize=13)
        ax.set_ylabel("PM2.5 (µg/m³)")
        ax.legend(fontsize=7, ncol=2, loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No PM2.5 data to plot.")
else:
    print("DataFrame is empty.")

---
## 📌 Notes

| Item | Detail |
|---|---|
| API docs | https://docs.openaq.org |
| PM2.5 parameter ID | `2` |
| PM10  parameter ID | `1` |
| Rate limit | Free tier ≈ 60 req/min — `SLEEP_SEC=1.2` keeps at ~50/min |
| 429 handling | Wait `SLEEP_429=30s`, retry up to `RETRY_MAX=3` times |
| 500 handling | Skip gracefully — means OpenAQ has no data for that sensor/period |
| `value` | Hourly **mean** concentration in µg/m³ |
| `count` | Number of sub-hourly measurements averaged |
| Outputs (Colab) | `/content/Delhi_OpenAQ/` + auto-downloaded |
| Outputs (local) | `Delhi_OpenAQ_Output/` next to the notebook |